In [36]:
import matplotlib
import platform
print ("Operating system: ", platform.system())
if "Linux" in platform.system():
    %matplotlib tk
else:
    %matplotlib qt    

import matplotlib.pyplot as plt
%autosave 180
%load_ext autoreload
%autoreload 2
import numpy as np

from scipy import ndimage as ndi
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
import scipy
import os
import time

#
from calibration.calibration import BMICalibration
from drift.drift import make_template, compute_drift_multi_frames, correct_drift, correct_drift_single_frame
from utils.utils import smooth_ca_time_series, compute_dff0



Operating system:  Linux


Autosaving every 180 seconds
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [37]:
#######################################################
#######################################################
#######################################################

def make_shifts_random_walk(data, 
                           max_shift=20):
    
    #
    shifts = []
    vals = np.arange(-1,2,1)
    shifts.append([0,0])
    choice_array = np.concatenate((np.zeros(59),[-1]))
    choice_array = np.concatenate((choice_array,[1]))
    for k in range(data.shape[0]-1):
        
        while True:
            x = np.random.choice(choice_array)
            y = np.random.choice(choice_array)
            
            if abs(x+shifts[k][0])<20:
                if abs(y+shifts[k][1])<20:
                    shifts.append([x+shifts[k][0],
                                   y+shifts[k][1]])
                    break
                    
    return np.int32(shifts)
    

In [38]:
###########################################################
############## MAKE SIMULATED DRIFT DATA FIRST ############
###########################################################

#
fname = '/home/cat/data/donato/bscope_tests/8499/data/Image_001_001_original.raw'
bmi_c = BMICalibration(fname)

#
shifts = make_shifts_random_walk(bmi_c.data)
print ("Shifts: ", shifts.shape)

plt.figure()
t=np.arange(len(shifts))/30
plt.plot(t,shifts[:,0], label='xshifts')
plt.plot(t,shifts[:,1], label='yshifts')
plt.xlabel("Time (sec)",fontsize=20)
plt.ylabel("Pixels",fontsize=20)
plt.legend(fontsize=20)
plt.title(fname)
plt.show()

#
from tqdm import trange
bmi_c.data_shifted = np.zeros(bmi_c.data.shape, dtype='uint16')
for k in trange(bmi_c.data.shape[0]):
    temp = correct_drift_single_frame(bmi_c.data[k], 
                                               shifts[k])
    #print ((temp-bmi_c.data[k]).sum())
    bmi_c.data_shifted[k] = temp
    
bmi_c.data_shifted.tofile(fname.replace('_original.raw','_shifted.raw'))
np.save(fname.replace('.raw','_shifts_xy.npy'), shifts)
print ("DONE...")

memmap :  (10000, 512, 512)
Shifts:  (10000, 2)


100%|████████████████████████████████████| 10000/10000 [00:12<00:00, 806.77it/s]


In [7]:
# plt.figure()
# ax=plt.subplot(121)
# plt.imshow(bmi_c.data_shifted[8000])
# ax=plt.subplot(122)
# plt.imshow(bmi_c.data[8000]-bmi_c.data_shifted[8000])
# plt.show()

In [8]:
#######################################################################
########### LOAD PRE-BMI DATA (e.g. 10-15mins recording) ##############
#######################################################################
fname_original = '/home/cat/data/donato/bscope_tests/8499/data/Image_001_001_original.raw'
fname_shifted = '/home/cat/data/donato/bscope_tests/8499/data/Image_001_001_shifted.raw'

# 
bmi_c = BMICalibration(fname_original)
bmi_c.fname_original = fname_original
bmi_c.fname_shifted = fname_shifted
bmi_c.smooth_ca_time_series = smooth_ca_time_series
bmi_c.compute_dff0 = compute_dff0

#
bmi_c.subsample = 10 # for std computation downsample to every N'th frame; the more frames the better the rois;
                  #   TODO: use correlation instead?! might be much faster; it is quit fast in other implemenations


memmap :  (10000, 512, 512)


In [9]:
####################################################################
############ MAKE TEMPLATE WITH SHIFTED OR ORIGINAL ################
####################################################################

#
print ("data: ", bmi_c.data.shape)

# 
n_iter = 1
n_imgs_to_sample = 2000
random_img_sample_flag = False
n_best_imgs = 100
plotting = True
n_cores = 8
template=None
idx_imgs = None

#
fname_template = bmi_c.fname_shifted

#
for k in range(n_iter):
    idx_imgs = None
    corr_maxs, template, idx_imgs = make_template(
                                              bmi_c.data, 
                                              fname_template,
                                              n_imgs_to_sample,
                                              n_best_imgs,
                                              template,
                                              idx_imgs,
                                              random_img_sample_flag,
                                              plotting,
                                              n_cores)
    

plt.figure()
plt.subplot(1, 2, 1)
print (bmi_c.data.shape, idx_imgs.shape)
temp = bmi_c.data[idx_imgs].mean(0)
plt.title("Average map over " + str(idx_imgs.shape[0]) + " images")
plt.imshow(temp,
           #vmin=0,
           #vmax=1500
           #
           )

#
plt.subplot(1, 2, 2)
plt.title("Average map over highest correlated " + str(n_best_imgs) + " images")
plt.imshow(template,
           #vmin=0,
           #vmax=1500
           )

#
plt.show()

#
bmi_c.template = template.copy()


data:  (10000, 512, 512)


100%|█████████████████████████████████████████████| 8/8 [00:10<00:00,  1.26s/it]


(10000, 512, 512) (2000,)


In [12]:
######################################################################
### COMPUTE MOTION CORRECTION FOR BOTH ORIGINAL VIDEO AND SHIFTED ####
######################################################################

#
n_cores = 8
subsample = 3   # subsample every N'th frame for drift to speed up;
                # TODO: would be good to use all frames!!

# align the artificailly shifted data not the raw
# # also extract shifts in raw data without aritifical drift
# fname_original = fname.replace('_original.raw','.raw')
# bmi_c.data_shifted = np.memmap(fname, dtype='uint16', mode='r').reshape(-1, 512, 512)

  
#

# 
bmi_c.fname = '/home/cat/data/donato/bscope_tests/8499/data/Image_001_001_shifted.raw'
shifts_drift, _ = compute_drift_multi_frames(0,
                                                     bmi_c,
                                                     subsample,
                                                     n_cores,)

bmi_c.fname = '/home/cat/data/donato/bscope_tests/8499/data/Image_001_001_original.raw'
shifts_original, _ = compute_drift_multi_frames(0,
                                                bmi_c,
                                                        subsample,
                                                        n_cores,)



computing motion on:  /home/cat/data/donato/bscope_tests/8499/data/Image_001_001_shifted.raw


100%|█████████████████████████████████████████████| 8/8 [00:17<00:00,  2.21s/it]

TODO: undo interpolation for drift with better function
computing motion on:  /home/cat/data/donato/bscope_tests/8499/data/Image_001_001_original.raw



100%|█████████████████████████████████████████████| 8/8 [00:19<00:00,  2.46s/it]

TODO: undo interpolation for drift with better function


In [15]:
# IF simualted drift data check against ground truth

fname_drift_xy = bmi_c.fname_original.replace('.raw','_shifts_xy.npy')
shifts_gt = np.load(fname_drift_xy)[:shifts_original.shape[0]]
print (shifts_gt.shape)

# # also extract shifts in raw data without aritifical drift
# fname_original = fname.replace('.raw','_original.raw')
# bmi_c.data_original = np.memmap(fname, dtype='uint16', mode='r').reshape(-1, 512, 512)

# # compute shifts on the original raw data without artificial drift
# shifts_raw, corr_maxs = compute_drift_multi_frames(bmi_c.data_original,
#                                                bmi_c.fname,
#                                                template,
#                                                subsample,
#                                                n_cores,)


plt.figure()
t=np.arange(shifts_drift.shape[0])/30
print (shifts_gt.shape,
       shifts_drift.shape,
       shifts_original.shape)

#
ax=plt.subplot(211)
#plt.plot(t,-shifts_drift[:,0], label='xshifts: real+simulated')
#plt.plot(t,-shifts_original[:,0], label='xshifts: real')
plt.plot(t,-(shifts_drift[:,0]-shifts_original[:,0]), label='xshifts: (real + simulated)-real')
plt.plot(t,shifts_gt[:,0]+1, label='xshifts: simulated')
plt.xlabel("Time (sec)",fontsize=20)
plt.ylabel("Pixels",fontsize=20)
plt.legend(fontsize=20)
plt.ylim(-40,40)

#
ax=plt.subplot(212)
plt.plot(t,-(shifts_drift[:,1]-shifts_original[:,1]), label='yshifts: (real + simulated)-real')
plt.plot(t,shifts_gt[:,1]+1, label='yshifts: simulated')
plt.xlabel("Time (sec)",fontsize=20)
plt.ylabel("Pixels",fontsize=20)
plt.legend(fontsize=20)
plt.ylim(-40,40)

#
plt.suptitle(bmi_c.fname_original)
plt.show()



(10000, 2)
(10000, 2) (10000, 2) (10000, 2)


In [5]:
####################################################
############# CORRECT / SHIFT DATA #################
####################################################

# correct self.data using deteted shifts above
bmi_c.data = correct_drift(bmi_c.data, shifts)


fixing drift calibration data: 100%|████| 20000/20000 [00:10<00:00, 1848.42it/s]


memmap :  (10000, 512, 512)
Shifts:  (10001, 2)


100%|███████████████████████████████████| 10000/10000 [00:05<00:00, 1926.94it/s]


In [34]:
######################################################################
######################################################################
######################################################################
# recompute shifts on the orignial raw data
# fname_2 = '/home/cat/data/donato/bscope_tests/8499/data/Image_001_001.raw'
# bmi_c.data = np.memmap(fname_2, 
#                        dtype='uint16', 
#                        mode='r')

# load online saved data:
data = np.load('/home/cat/data/donato/bscope_tests/8499/databmi_results.npz', allow_pickle=True)
shifts_online = data['drift_array']

#
data = np.load('/home/cat/data/donato/bscope_tests/8499/rois_pixels_and_thresholds.npz',allow_pickle=True)
template = data['calibration_template']


#

# 
fname_original = '/home/cat/data/donato/bscope_tests/8499/data/Image_001_001.raw'
bmi_c = BMICalibration(fname_original)
bmi_c.template = template.copy()

#
subsample = 3
n_cores = 8
shifts_original, _ = compute_drift_multi_frames(0,
                                                bmi_c,
                                                subsample,
                                                n_cores,)

#
#abs_ = np.abs(shifts_online)


memmap :  (10000, 512, 512)
computing motion on:  /home/cat/data/donato/bscope_tests/8499/data/Image_001_001.raw


100%|█████████████████████████████████████████████| 8/8 [00:36<00:00,  4.56s/it]

TODO: undo interpolation for drift with better function


In [35]:
####
if False:
    print ('shifts online: ', shifts_online.shape)
    # remove uggliest jumps
    x_online = shifts_online[:,0].copy()
    threshold=10
    for k in range(1,x_online.shape[0],1):
        if abs(x_online[k]-x_online[k-1])>threshold:
            x_online[k] = x_online[k-1]

#
plt.figure()
plt.plot(shifts_original[:,0], label="OFFLINE: original - computed motion")
#plt.plot(np.cumsum(x_online), label = "ONLINE: original - computed motion")
plt.legend()
#plt.ylim(-50,50)
plt.show()